# ZenteiQ AI Tech Innovations — Task 2: Dense Model (Qwen)
## MaxText Qwen3-0.6B GPU Training Run & Benchmark

This notebook provides a detailed setup and execution workflow for training the **Qwen3-0.6B** base model on a **GPU-only** environment using **MaxText** and **JAX** with synthetic data.

### 1. Installation & Environment Setup

We will clone the MaxText repository and install its dependencies. When running on a GPU-only environment (such as Google Colab with the GPU runtime selected). We will install the dependencies using `uv` for fast installations.

In [1]:
# Clone the MaxText repository
!git clone https://github.com/AI-Hypercomputer/maxtext.git
%cd maxtext

# Install uv (fast package installer)
!pip install uv

# Install MaxText GPU dependencies
!uv pip install -e .[cuda12]

# Restart the runtime after this step if run in Colab!

Cloning into 'maxtext'...
remote: Enumerating objects: 97091, done.
remote: Counting objects: 100% (2308/2308), done.
remote: Compressing objects: 100% (1169/1169), done.
remote: Total 97091 (delta 1873), reused 1179 (delta 1139), pack-reused 94783 (from 3)
Receiving objects: 100% (97091/97091), 411.58 MiB | 20.68 MiB/s, done.
Resolving deltas: 100% (70853/70853), done.
/content/maxtext
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 21.4 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.13 environment at: /usr
Resolved 252 packages in 3.01s                                       
Prepared 116 packages in 4m 21s                                          
Uninstalled 54 packages in 553ms
Installed 117 packages in 439ms                             
 - absl-py==1.4.0
 + absl-py==2.4.0
 - aiofiles==24.1.0
 + aiofiles==25.1.0
 + aqtp==0.9.0
 + astroid==4.0.4
 + auditwheel==6.7.0
 + black==25.12.0
 + build==1.5.0
 + cfgv==3.5.0
 + chex==0.1.92
 + cloud-accelerator-diagnostics==0.1.1
 + cl

In [2]:
!uv pip install qwix -q

In [3]:
!uv pip install tokamax -q

### 2. Verify JAX GPU Backend Connection

Run the following cell to confirm that JAX successfully detects the GPU backend.

In [5]:
import jax
print("JAX Version:", jax.__version__)
print("Available devices:", jax.devices())
print("Default backend:", jax.default_backend())

JAX Version: 0.10.2
Available devices: [CudaDevice(id=0)]
Default backend: gpu


### 3. Configuration Breakdown

Below are the parameters we are setting for this training run, along with their purpose and effect:

| Configuration Parameter | Value | Purpose | Effect |
| :--- | :--- | :--- | :--- |
| **`model_name`** | `qwen3-0.6b` | Specifies the base architecture layout to use. | Loads structural config from `configs/models/qwen3-0.6b.yml` setting embedding dimensions (`base_emb_dim: 1024`), intermediate MLP dimension (`base_mlp_dim: 3072`), attention query/KV heads (`16`/`8`), and number of layers (`28`). |
| **`steps`** | `50` | Defines training duration. | Runs the optimization/training loop for exactly 50 iterations, which is sufficient to compile the graph, calculate device step times, and measure TFLOPs throughput. |
| **`dataset_type`** | `synthetic` | Determines the data loader pipeline format. | Bypasses external dataset downloading and disk I/O bottlenecks by dynamically generating random tensors of input tokens on GPU memory. This allows pure computational throughput benchmarking. |
| **`run_name`** | `qwen3-0.6b-gpu` | Uniquely names the specific execution run. | Prevents overwriting other runs and creates nested folders named `qwen3-0.6b-gpu/` for output checkpoints and statistics. |

### 4. Execute GPU Training (Qwen 0.6B)

We run the main training script with our configuration overrides. We capture the stdout logs to parse metrics later.

In [14]:
!cat src/maxtext/configs/base.yml

# Copyright 2023–2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#    https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# This sentinel is a reminder to choose a real run name.
# If there is already a checkpoint under this run, that checkpoint will auto-resume.
#
# NOTE: Some unit/integration tests in MaxText do not always run this file directly.
# When running in decoupled mode (DECOUPLE_GCLOUD=TRUE), tests may use
# `decoupled_base_test.yml` instead of `base.yml` via `tests/utils/test_helpers.py`.
run_name: ""

model_name: "default"

In [6]:
!sudo apt-get update && sudo apt-get install -y \
    libxcomposite1 \
    libxcursor1 \
    libxdamage1 \
    libxext6 \
    libxfixes3 \
    libxi6 \
    libxrandr2 \
    libxrender1 \
    libxtst6 \
    libgbm1 \
    libasound2



Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]       
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,764 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,062 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,

In [5]:
# Run training for 50 steps on GPU
!python3 -m maxtext.trainers.pre_train.train src/maxtext/configs/base.yml \
    hardware=gpu \
    model_name=qwen3-0.6b \
    steps=50 \
    dataset_type=synthetic \
    base_output_directory=$(pwd)/gpu-output-v2 \
    run_name=qwen3-0.6b-gpu \
    per_device_batch_size=1 \
    attention=dot_product \
    max_target_length=1024 \
    enable_checkpointing=false

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2026-06-18 05:09:08.516128: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[transformers] DeepseekV32Config got `key=rope_scaling` in kwargs but hasn't set it as attribute. For RoPE standardization you need to set `self.rope_parameters` in model's config. 
I0618 05:09:27.829763 139734748279936 max_utils.py:259] Attempting to initialize the jax distributed system for GPU backend...
I0618 05:09:27.830017 139734748279936 max_utils.py:261

In [7]:
!uv pip install aqtp ml-goodput-measurement ml-collections

Using Python 3.12.13 environment at: /usr
Checked 3 packages in 98ms


In [8]:
!uv pip install pathwaysutils aqt

Using Python 3.12.13 environment at: /usr
Resolved 68 packages in 361ms                                        
Prepared 9 packages in 8.19s                                             
Installed 9 packages in 61ms                                
 + anki==26.5
 + aqt==26.5
 + pyqt6==6.11.0
 + pyqt6-qt6==6.11.1
 + pyqt6-sip==13.11.1
 + pyqt6-webengine==6.11.0
 + pyqt6-webengine-qt6==6.11.1
 + truststore==0.10.4
 + waitress==3.0.2


In [9]:
!uv pip install -U transformers

Using Python 3.12.13 environment at: /usr
Resolved 27 packages in 218ms                                        
Prepared 7 packages in 722ms                                             
Uninstalled 7 packages in 543ms
Installed 7 packages in 56ms                                
 - anyio==4.13.0
 + anyio==4.14.0
 - certifi==2026.5.20
 + certifi==2026.6.17
 - filelock==3.29.2
 + filelock==3.29.4
 - fsspec==2026.4.0
 + fsspec==2026.6.0
 - huggingface-hub==1.18.0
 + huggingface-hub==1.19.0
 - tqdm==4.67.3
 + tqdm==4.68.3
 - transformers==5.10.2
 + transformers==5.12.1


In [10]:
!uv pip install drjax -q

In [15]:
!uv pip install --force-reinstall nvidia-cudnn-cu12 nvidia-cublas-cu12 nvidia-cuda-runtime-cu12

Using Python 3.12.13 environment at: /usr
Resolved 4 packages in 88ms                                          
Prepared 4 packages in 0.74ms                                            
Uninstalled 4 packages in 3ms
Installed 4 packages in 4mscu12==12.9.79                    
 ~ nvidia-cublas-cu12==12.9.2.10
 ~ nvidia-cuda-nvrtc-cu12==12.9.86
 ~ nvidia-cuda-runtime-cu12==12.9.79
 ~ nvidia-cudnn-cu12==9.23.2.1


In [11]:

!ls

drive  gpu-output-v2  maxtext  sample_data


### 5. Parse Metrics & Logs

MaxText prints metric logs regularly during training. We will extract the compilation time, step times, loss, learning rate, and TFLOPs using a helper script.

In [19]:
%load_ext tensorboard


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [18]:
!uv pip install --upgrade setuptools tensorboard


Using Python 3.12.13 environment at: /usr
Resolved 13 packages in 194ms                                        
Prepared 1 package in 6ms                                                
Uninstalled 1 package in 4ms
Installed 1 package in 11ms                                 
 - protobuf==6.33.6
 + protobuf==7.35.1


In [21]:
# %tensorboard --logdir gpu-output-v2/qwen3-0.6b-gpu/tensorboard/


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!ls 

AUTHORS      CONTRIBUTING.md  LICENSE_HEADER  pyproject.toml  scripts  tools
benchmarks   docs	      PREFLIGHT.md    pytest.ini      src
codecov.yml  LICENSE	      pylintrc	      README.md       tests


In [12]:
!cp -r gpu-output-v2 /content/drive/MyDrive/


In [2]:
import os
import sys

# Detect Python's nvidia-cudnn library path
python_version = f"python{sys.version_info.major}.{sys.version_info.minor}"
site_packages_path = f"/usr/local/lib/{python_version}/dist-packages/nvidia/cudnn/lib"

if os.path.exists(site_packages_path):
    os.environ["LD_LIBRARY_PATH"] = site_packages_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    print(f"Set LD_LIBRARY_PATH to: {os.environ['LD_LIBRARY_PATH']}")
else:
    # If using virtual environments, check sys.prefix
    site_packages_path = f"{sys.prefix}/lib/{python_version}/site-packages/nvidia/cudnn/lib"
    if os.path.exists(site_packages_path):
        os.environ["LD_LIBRARY_PATH"] = site_packages_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")
        print(f"Set LD_LIBRARY_PATH to: {os.environ['LD_LIBRARY_PATH']}")
    else:
        print("Could not automatically locate pip's nvidia-cudnn path. Try Fix 2.")


Set LD_LIBRARY_PATH to: /usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib:/usr/lib64-nvidia
